# Smart Book Assistant Agent

In [1]:
import chromadb
from google import genai
import pandas as pd

In [8]:
client = genai.Client(api_key="Write_Here_Your_Own_API_Address")

In [3]:
chroma_client = chromadb.PersistentClient(path="./book_agent_db")
collection = chroma_client.get_or_create_collection(name="book_collection")

In [4]:
def search_book_database(query: str) -> str:
    """Kitap veritabanında arama yapar ve en uygun kitap açıklamalarını ve meta verilerini döndürür."""
    results = collection.query(query_texts=[query], n_results=3)
    
    # Bulunan kitapları birleştirip ajan için bağlam (context) oluşturma
    context_blocks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        block = f"Title: {meta['title']} | Author: {meta['authors']} | Rating: {meta['rating']} | Summary: {doc}"
        context_blocks.append(block)
        
    return "\n\n".join(context_blocks)

In [5]:
def run_book_agent(user_goal: str, max_retries=3):
    print(f"\n[Agent Goal]: {user_goal}")
    retrieved_data = search_book_database(user_goal)

    prompt = f"""
    You are an autonomous AI Book Assistant and Literary Expert. 
    Your task is to help the user based ONLY on the retrieved library database below.
    
    Retrieved Library Data:
    {retrieved_data}
    
    User Request: {user_goal}
    
    Provide a professional, engaging, and structured response recommending the best books with details.
    """

    for attempt in range(max_retries):
        try:
            print(f"[Attempt {attempt+1}] Generating response via Gemini...")
            response = client.models.generate_content(
                model="gemini-3.8-flash", contents=prompt
            )
            return response.text
        except Exception as e:
            print(f"Error encountered: {e}")
            if attempt < max_retries - 1:
                print("Waiting 3 seconds before retrying...")
                time.sleep(3)
            else:
                return "The server is currently busy. Please try running the cell again in a moment."

In [6]:
agent_response = run_book_agent("Can you recommend a captivating fantasy or dystopian book for me?")


[Agent Goal]: Can you recommend a captivating fantasy or dystopian book for me?
[Attempt 1] Generating response via Gemini...


In [7]:
print("\n[Agent Final Output]:\n", agent_response)


[Agent Final Output]:
 Here are top recommendations from our library database tailored to your interest in **captivating fantasy** and **dystopian/apocalyptic** fiction:

---

### **For Dystopian & Apocalyptic Fiction:**

#### **1. *The Stand***
* **Author:** Stephen King (Illustrated by Bernie Wrightson)
* **Rating:** 4.34 / 5
* **Overview:** 
  A masterwork of apocalyptic fiction, *The Stand* portrays a world devastated by a biological catastrophe. After a patient escapes a testing facility carrying a mutated strain of super-flu, 99 percent of the global population is wiped out within weeks. 
* **Why You Should Read It:** 
  The narrative shifts into an elemental clash between good and evil among the frightened survivors. They are drawn to two vastly different leaders: Mother Abagail, a 108-year-old woman striving to build a peaceful community in Colorado, and Randall Flagg, the sinister "Dark Man" who thrives on chaos and violence. It offers a deeply riveting, eerily plausible stor